# 002 · itertools: Combinatorics and Grouping

> **Standard Library › Functional Tools** · Python 3.14

The second half of `itertools` covers **combinatorics** (every pair, every ordering, every subset), **grouping** consecutive items, **running totals**, and **splitting** one iterator into several. These replace hand-written nested loops and the "scan" operator that LINQ never shipped.

**In this notebook you will learn:**

- `product`, `permutations`, `combinations`, `combinations_with_replacement` — and when to use which
- `groupby`, why it needs **sorted** input (the big difference from LINQ `GroupBy`), and why its groups disappear
- `accumulate` for running totals (a "Scan" / running `Aggregate`)
- `tee` for splitting one iterator into independent copies

## .NET mental model

| Need | C# / .NET | Python |
|---|---|---|
| Cartesian product | nested `from ... from ...` / `SelectMany` | `product(a, b)` |
| Orderings / subsets | no BCL support — hand-written recursion | `permutations`, `combinations`, `combinations_with_replacement` |
| Group by key | `GroupBy(key)` — groups **all** items, any order | `groupby(sorted_items, key)` — groups **consecutive** items |
| Running total | no BCL `Scan` (System.Interactive has one) | `accumulate(it, func)` |
| Final fold | `Aggregate(seed, func)` | `functools.reduce` *(see Standard Library › Functional Tools › 003 · functools)* |
| Enumerate twice | re-enumerate the `IEnumerable<T>` | `tee(it, n)` or `list(it)` |

All of these are lazy, like the rest of `itertools`. Combinatoric results can be **huge** — always think about the size before calling `list()`.

## `product` — nested loops in one call

```csharp
var cells = from size in new[] { "S", "M", "L" }
            from color in new[] { "red", "blue" }
            select (size, color);
```

`product(a, b)` yields every `(x, y)` pair — exactly what two nested `for` loops would produce, in the same order (rightmost input varies fastest).

In [ ]:
from itertools import product

sizes = ["S", "M", "L"]
colors = ["red", "blue"]
for size, color in product(sizes, colors):
    print(size, color, end=" | ")

`repeat=n` multiplies an input by itself — e.g. every 3-bit pattern, or every roll of two dice:

In [ ]:
print(["".join(bits) for bits in product("01", repeat=3)])

two_dice = list(product(range(1, 7), repeat=2))
print(len(two_dice), "outcomes;", sum(1 for a, b in two_dice if a + b == 7), "sum to 7")

**Tip:** `product` flattens arbitrarily deep nesting. Five nested loops become one `for` with a 5-tuple — no "arrow code".

## `permutations` — orderings

`permutations(it, r)` yields every **ordered** arrangement of `r` items (default: all of them). Order matters, items are not reused.

In [ ]:
from itertools import permutations

print(list(permutations("ABC")))              # 3! = 6 orderings
print(["".join(p) for p in permutations("ABC", 2)])   # 3P2 = 6 ordered pairs

## `combinations` — subsets where order doesn't matter

`combinations(it, r)` yields each **unordered** selection of `r` items once: `('A', 'B')` appears, `('B', 'A')` does not. A classic use is "every pair" without the `i < j` double loop.

```csharp
// C# — the manual way
for (int i = 0; i < items.Length; i++)
    for (int j = i + 1; j < items.Length; j++)
        Compare(items[i], items[j]);
```

In [ ]:
from itertools import combinations

players = ["alice", "bob", "carol", "dave"]
for a, b in combinations(players, 2):          # every match in a round-robin
    print(f"{a} vs {b}")

`combinations_with_replacement` allows the same item to be picked again (still unordered) — e.g. all ways to pick 2 scoops from 3 flavours:

In [ ]:
from itertools import combinations_with_replacement

flavours = ["vanilla", "choc", "mint"]
print(list(combinations_with_replacement(flavours, 2)))

### Which one do I need?

| Function | Order matters? | Reuse items? | Count for n=4, r=2 |
|---|---|---|---|
| `product(xs, repeat=r)` | yes | yes | 4² = 16 |
| `permutations(xs, r)` | yes | no | 4·3 = 12 |
| `combinations(xs, r)` | no | no | C(4,2) = 6 |
| `combinations_with_replacement(xs, r)` | no | yes | 10 |

`math.perm` and `math.comb` compute these counts without generating anything *(see Standard Library › Numbers & Math › 001 · The math Module)*.

In [ ]:
import math

xs = "ABCD"
print(len(list(product(xs, repeat=2))), len(list(permutations(xs, 2))),
      len(list(combinations(xs, 2))), len(list(combinations_with_replacement(xs, 2))))
print(math.perm(4, 2), math.comb(4, 2))

⚠️ **Gotcha:** uniqueness is by **position**, not by value. If the input contains duplicates, so will the output. Wrap in `set(...)` if you need distinct values.

In [ ]:
print(list(permutations("AAB")))          # 6 tuples, several identical
print(sorted(set(permutations("AAB"))))   # 3 distinct orderings

## `groupby` — group **consecutive** items

This is the biggest trap for LINQ users.

```csharp
var byCategory = products.GroupBy(p => p.Category);   // collects ALL items per key
```

LINQ's `GroupBy` looks at the whole sequence and builds one group per distinct key. `itertools.groupby` is a **streaming** operator: it starts a new group whenever the key **changes** from one item to the next. It behaves like the Unix `uniq` command.

In [ ]:
from itertools import groupby

categories = ["fruit", "fruit", "veg", "fruit", "veg", "veg"]
for key, group in groupby(categories):
    print(key, list(group))

⚠️ **Gotcha:** `"fruit"` appears twice above because the input wasn't sorted. To get LINQ-style grouping, **sort by the same key first**:

In [ ]:
products = [
    ("apple", "fruit"), ("carrot", "veg"), ("banana", "fruit"),
    ("leek", "veg"), ("cherry", "fruit"),
]

def by_category(p):
    return p[1]

for category, items in groupby(sorted(products, key=by_category), key=by_category):
    print(category, [name for name, _ in items])

**Tip:** if you just want a "key → list" dictionary and don't care about streaming, a `defaultdict(list)` is usually clearer and doesn't need sorting *(see Standard Library › Data Structures › 003 · defaultdict, OrderedDict and ChainMap)*:

In [ ]:
from collections import defaultdict

grouped = defaultdict(list)
for name, category in products:
    grouped[category].append(name)
print(dict(grouped))

Consecutive grouping is genuinely useful on its own — for example, run-length encoding or detecting streaks:

In [ ]:
signal = "AAABBCAAAA"
print([(ch, len(list(run))) for ch, run in groupby(signal)])   # run-length encoding

results = ["W", "W", "L", "W", "W", "W", "L"]
print("longest win streak:", max(len(list(g)) for k, g in groupby(results) if k == "W"))

⚠️ **Gotcha:** each `group` is a **lazy view into the shared underlying iterator**. As soon as `groupby` advances to the next key, the previous group is gone. If you keep groups for later, materialise them with `list()` inside the loop.

In [ ]:
words = ["ant", "ape", "bee", "bat", "cat"]

# WRONG: store the group iterators, consume them later
kept = {k: g for k, g in groupby(words, key=lambda w: w[0])}
print({k: list(g) for k, g in kept.items()})     # all empty!

# RIGHT: materialise each group while it is current
kept = {k: list(g) for k, g in groupby(words, key=lambda w: w[0])}
print(kept)

## `accumulate` — running totals (Scan)

`Aggregate` in LINQ gives you only the **final** result. `accumulate` yields **every intermediate** result — what reactive/functional libraries call `Scan`.

```csharp
// C# — no BCL Scan; you write a loop or use System.Interactive's Scan
int running = 0;
var totals = sales.Select(s => running += s);
```

By default it adds; pass any two-argument function to change the operation.

In [ ]:
from itertools import accumulate
import operator

sales = [5, 3, 8, 2, 7]
print(list(accumulate(sales)))                    # running sum: [5, 8, 16, 18, 25]
print(list(accumulate(sales, max)))               # running max: [5, 5, 8, 8, 8]
print(list(accumulate(sales, operator.mul)))      # running product

`initial=` supplies a seed (like `Aggregate(seed, ...)`), which also becomes the first output:

In [ ]:
balance_changes = [+100, -30, -50, +20]
print(list(accumulate(balance_changes, initial=1000)))   # [1000, 1100, 1070, 1020, 1040]

**Tip:** if you only want the final value, `sum()` or `functools.reduce` is the true `Aggregate`. `accumulate` is for when you need the whole history (balances, cumulative counts, prefix sums).

## `tee` — split one iterator into several

Because an iterator can only be consumed once, you can't easily walk it twice. `tee(it, n)` returns `n` independent iterators that each see the full stream. Internally it buffers the items that one copy has seen but another hasn't.

In [ ]:
from itertools import tee

stream = iter([3, 1, 4, 1, 5])
a, b = tee(stream, 2)
print(list(a))   # [3, 1, 4, 1, 5]
print(list(b))   # [3, 1, 4, 1, 5] — independent copy

⚠️ **Gotcha:** after calling `tee`, **don't use the original iterator** — advancing it bypasses the copies' buffer. And if one copy runs far ahead of the other, `tee` buffers everything in between; at that point `list(it)` is simpler and just as memory-hungry.

In [ ]:
stream = iter("abcde")
a, b = tee(stream)
next(stream)                 # oops — consumed 'a' behind tee's back
print(list(a), list(b))      # both copies are missing 'a'

A common legitimate use is looking ahead by one item (this is roughly how `pairwise` works):

In [ ]:
def with_next(iterable):
    current, ahead = tee(iterable)
    next(ahead, None)                      # shift the second copy by one
    return zip(current, ahead)

print(list(with_next("abcd")))

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Nested loops / cross join | `from a in A from b in B` | `product(A, B)` |
| Self cross join | `from a in A from b in A` | `product(A, repeat=2)` |
| All orderings | hand-written | `permutations(xs, r)` |
| All unordered pairs | `for i / for j > i` | `combinations(xs, 2)` |
| Subsets with repeats | hand-written | `combinations_with_replacement(xs, r)` |
| Group by key | `.GroupBy(k)` | `groupby(sorted(xs, key=k), key=k)` |
| Run-length / streaks | hand-written | `groupby(xs)` (unsorted) |
| Running total | System.Interactive `Scan` | `accumulate(xs)` |
| Seeded running fold | `Scan(seed, f)` | `accumulate(xs, f, initial=seed)` |
| Iterate twice | re-enumerate | `tee(it)` or `list(it)` |

## Exercises

1. Print every combination of two distinct toppings from `["ham", "olive", "pepper", "onion"]` (6 lines).
2. Given `logs = ["ERROR", "ERROR", "INFO", "ERROR", "INFO", "INFO", "INFO"]`, count each level **in total** using `groupby` (expect `ERROR 3`, `INFO 4`).
3. Given daily step counts `[4000, 6500, 3000, 8000]`, print the running total and the running maximum side by side.
4. Using `product`, find all `(x, y)` with `x, y` in `range(1, 10)` where `x * y == 12` and `x <= y`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from itertools import combinations, groupby, accumulate, product

# 1 — unordered, no repeats
for a, b in combinations(["ham", "olive", "pepper", "onion"], 2):
    print(a, "+", b)

# 2 — sort first, otherwise ERROR/INFO appear in several groups
logs = ["ERROR", "ERROR", "INFO", "ERROR", "INFO", "INFO", "INFO"]
for level, group in groupby(sorted(logs)):
    print(level, sum(1 for _ in group))

# 3 — two accumulate calls zipped together
steps = [4000, 6500, 3000, 8000]
for total, best in zip(accumulate(steps), accumulate(steps, max)):
    print(total, best)

# 4 — product replaces the nested loops
print([(x, y) for x, y in product(range(1, 10), repeat=2) if x * y == 12 and x <= y])
# [(2, 6), (3, 4)]
```

</details>